# Joint Zero-Shot Motion Estimation and Iterative Reconstruction for Motion-Corrected Cardiac T1 Mapping

Cardiac T1 mapping suffers from cardiac motion: the standard clinical strategy is to
synchronize the acquisition to a fixed cardiac phase (ECG-triggering), which limits data
collection to a small fraction of each heartbeat and thus leads to long scan times and low
scan efficiency. Here we instead consider data acquired **continuously over the full cardiac
cycle**, which is far more efficient but requires estimating and correcting the complete
non-rigid cardiac deformation from highly undersampled data.

The proposed method obtains motion-corrected T1 maps by **coupling motion estimation, image
reconstruction and parametric fitting in a single iterative scheme**, instead of solving them
sequentially.

This notebook is a clean, self-contained walkthrough of the method implemented in
[`main_x_cat.py`](../main_x_cat.py). It runs the full pipeline on the simulated XCAT
phantom and ends with a plot of the estimated motion-corrected T1 map next to the reference
true map.

---

## Outline

0. [Method](#0) — problem formulation and proposed joint optimization
1. [Setup](#1) — imports, device, parameters
2. [Simulated data](#2) — XCAT phantom, k-space simulation
3. [Operators and initial reconstruction](#3) — $\mathbf{x}_\text{m}$
4. [T1 estimator and No MoCo baseline](#4) — effect of uncorrected motion
5. [Motion-compensated reference](#5) — $\mathbf{x}_\text{r}$, $\mathbf{p}_\text{r}$
6. [Motion estimation](#6) — initial motion fields $\boldsymbol{\phi}, \boldsymbol{\phi}^{-1}$
7. [Initialization (iteration 0)](#7) — $\mathbf{x}^{(0)}$, $\mathbf{p}^{(0)}$, SVD basis $\mathbf{U}$
8. [Alternating optimization loop](#8)
9. [Final T1 map](#9)

---

<a id="0"></a>
## 0. Method

### Problem formulation

The measured k-space data $\mathbf{y}$ are related to the unknown tissue parameters
$\mathbf{p}$ through the acquisition model

$$
\mathbf{y} \;=\; \mathbf{A}\,\mathbf{M}_{\boldsymbol{\phi}}\,\mathbf{Q}(\mathbf{p}) \;+\; \boldsymbol{\epsilon},
$$

where $\boldsymbol{\epsilon}$ accounts for measurement noise and each operator describes a
different aspect of the acquisition:

- $\mathbf{Q}$ — the exponential signal model, mapping the $P$ quantitative parameters
  $\mathbf{p}$ to a dynamic $\mathbf{Q}(\mathbf{p}) = (Q_{t_1}(\mathbf{p}),\dots,Q_{t_T}(\mathbf{p}))$,
  i.e. a sequence of $T$ images with time-varying contrast;
- $\mathbf{M}_{\boldsymbol{\phi}}$ — the motion operator, applying the displacement fields
  $\boldsymbol{\phi} = (\phi_1,\dots,\phi_T)$ time-wise to a dynamic via local spatial
  interpolation kernels; since data are acquired continuously over multiple cardiac cycles,
  these fields describe the full range of heart motion, including diastolic and systolic phases;
- $\mathbf{A}$ — the forward operator, combining coil sensitivity maps with the non-Cartesian
  Fourier transform defined by the chosen undersampling trajectory.

Recovering the motion-corrected parameters from $\mathbf{y}$ then corresponds to solving

$$
\mathbf{p}_{\text{MoCo}},\ \boldsymbol{\phi}_{\text{cardiac}}
\;=\;
\arg\min_{\mathbf{p},\,\boldsymbol{\phi}}\;
\big\|\mathbf{A}\,\mathbf{M}_{\boldsymbol{\phi}}\,\mathbf{Q}(\mathbf{p}) - \mathbf{y}\big\|_2^2 .
$$

Solving this directly is not tractable: the cardiac motion fields are unknown and hard to
recover without intermediate reconstruction steps, the parameter estimation is non-linear
because of $\mathbf{Q}$, and the k-space data are highly undersampled, making the
reconstruction itself ill-posed.

### Proposed joint optimization

Introducing the dynamic $\mathbf{x}$ and the low-rank coefficients $\mathbf{c}_j$ as auxiliary
variables (variable splitting), the problem is reformulated into two subproblems: **(a)** motion
estimation and **(b)** motion-corrected reconstruction and parameter estimation, which are
solved in alternation and stay coupled through the outer iterations:

$$
\begin{cases}
\boldsymbol{\theta}^{*} \in \arg\min_{\boldsymbol{\theta}} \ \mathcal{L}\big(\boldsymbol{\phi}_{\boldsymbol{\theta}},\boldsymbol{\phi}^{-1}_{\boldsymbol{\theta}},\mathbf{x}_{\text{m}},\mathbf{x}_{\text{r}}\big) & \text{(a)} \\[6pt]
\mathbf{p}^{*},\mathbf{x}^{*},\mathbf{c}_j^{*} \in \arg\min_{\mathbf{p},\mathbf{x},\mathbf{c}_j} \ \mathcal{F}\big(\mathbf{p},\mathbf{x},\mathbf{c}_j,\boldsymbol{\phi}_{\boldsymbol{\theta}^{*}}\big) & \text{(b)}
\end{cases}
$$

**(a) Motion estimation.** Forward and inverse motion fields are the output of a CNN,
$(\boldsymbol{\phi}_{\boldsymbol{\theta}}, \boldsymbol{\phi}^{-1}_{\boldsymbol{\theta}}) = \mathbf{u}_{\boldsymbol{\theta}}(\mathbf{x}_{\text{m}})$,
whose weights are optimized **zero-shot** (no training data, no pre-trained weights: the network
is fitted to this single scan) and **unsupervised** (no ground-truth motion or T1 is ever used),
by minimizing

$$
\mathcal{L}(\boldsymbol{\phi},\boldsymbol{\phi}^{-1},\mathbf{x}_{\text{m}},\mathbf{x}_{\text{r}})
=
\underbrace{\frac{1}{2}\sum_{t=1}^{T}\Big[\mathrm{MI}\big(\mathbf{M}_{\phi_t}x^{\text{r}}_t,\,x^{\text{m}}_t\big)+\mathrm{MI}\big(\mathbf{M}_{\phi_t^{-1}}x^{\text{m}}_t,\,x^{\text{r}}_t\big)\Big]}_{\text{bidirectional mutual-information loss}}

+\;\underbrace{\frac{1}{2}\big\|\mathbf{A}\mathbf{M}_{\boldsymbol{\phi}}\mathbf{x}_{\text{r}}-\mathbf{y}\big\|_2^2}_{\text{k-space data consistency}}
\;+\;\underbrace{\sum_{t=1}^{T}\Big(\lambda\|\nabla\phi_t\|_2^2+\mu\|\mathrm{div}\,\phi_t\|_2^2\Big)}_{\text{regularization on motion}}
$$

where $\mathbf{x}_{\text{m}}$ is the moving (uncorrected) dynamic and $\mathbf{x}_{\text{r}}$ a
synthetic motion-compensated reference, obtained by fitting the signal model to a subset of
frames (indicating diastole) and progressively updated across the outer iterations.

**(b) Motion-corrected reconstruction and parameter estimation.** Given the current motion
fields, the functional

$$
\mathcal{F}(\mathbf{p},\mathbf{x},\mathbf{c}_j,\boldsymbol{\phi})
=
\underbrace{\frac{1}{2}\big\|\mathbf{A}\mathbf{M}_{\boldsymbol{\phi}}\mathbf{x}-\mathbf{y}\big\|_2^2}_{\text{data consistency}}
+\underbrace{\frac{\lambda_1}{2}\big\|\mathbf{x}-\mathbf{Q}(\mathbf{p})\big\|_2^2}_{\text{signal-model penalty}}
+\underbrace{\frac{\lambda_2}{2}\sum_j\Big(\|\mathbf{R}_j\mathbf{x}-\mathbf{U}\mathbf{c}_j\|_2^2+\lambda_3\|\mathbf{c}_j\|_1\Big)}_{\text{SVD-based low-rank + sparsity}}
$$

is minimized with $\lambda_1,\lambda_2,\lambda_3>0$. Here $\mathbf{U}$ is a fixed temporal
basis obtained from an SVD of the initial dynamic, and $\mathbf{R}_j$ extracts the temporal
signal at voxel $j$.

### Alternating scheme

Each outer iteration updates one block at a time, keeping the others fixed:

| Step | Variable | Solver |
|:--:|:--:|---|
| 0 | $\mathbf{x}_{\text{r}}$ | reference update: L-BFGS fit of the signal model on an expanding, ECG-selected subset of frames |
| 1 | $\boldsymbol{\phi},\ \boldsymbol{\phi}^{-1}$ | zero-shot unsupervised registration network (MI matching + data consistency + smoothness) |
| 2 | $\mathbf{c}_j$ | soft-thresholding (closed form) |
| 3 | $\mathbf{x}$ | conjugate gradient on the quadratic subproblem |
| 4 | $\mathbf{p}$ | L-BFGS fit of the constrained signal model, on the $\lfloor 0.85\,T\rfloor$ frames with smallest motion amplitude |

After $N_{\text{iter}}$ outer iterations (here $N_{\text{iter}} = 4$, plus one initialization
iteration), the scheme returns the final motion-corrected parametric maps
$\mathbf{p}_{\text{MoCo}}$, in particular the T1 map.

<a id="1"></a>
## 1. Setup

Imports, device and hyper-parameters. The parameters are read from
[`utils/runtime_parameters.py`](../utils/runtime_parameters.py) so that the notebook
stays in sync with the scripts.

In [ ]:
import sys
from pathlib import Path

# Make the repository root and src/ importable from inside notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path[:0] = [str(REPO_ROOT), str(REPO_ROOT / 'src')]

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

import mrpro

from data.loading_xcat import load_xcat_data
from operators.construct_recon_operators import construct_reconstruction_operators
from operators.MotionOperator import MotionOperator
from cardiac_motion.MotionEstimationWithInverse import MotionEstimationWithInverse
from parameters_estimation.T1ParameterEstimator import T1ParameterEstimator
from regularization.svd import compute_svd, update_svd
from utils.MaskImgSize import MaskImgSize
from utils.rr_threshold_functions import (
    find_valid_rr_threshold,
    update_rr_expansion_and_compute_rr_mask,
)
from configs.runtime_parameters import MOTION_NET_PARAMS, RECONSTRUCTION_PARAMS, RUN_SETTINGS
from visualization.colormaps import cmap_t1  # T1 colormap used throughout the paper figures

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

# The motion network is trained from a random initialization, so fix the seeds
# to make the run reproducible.
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
# XCAT phantom data (k-space, ground-truth maps) and the cardiac ROI mask parameters.
DATA_ROOT = REPO_ROOT / 'x_cat_study' / 'data' / 'updated_12-08-2026'
PATIENT_PREFIX = 'xcat_100bpm'

# Region of interest (heart), used for display, by the SVD prior and by the motion network.
roi_mask = MaskImgSize()
roi_mask.load_params(str(DATA_ROOT / f'mask_parameters_{PATIENT_PREFIX}.json'))
print(f'ROI mask: {roi_mask.get_masked_shape()} at (h={roi_mask.h_start}, w={roi_mask.w_start})')


n_outer = int(RECONSTRUCTION_PARAMS['opt_iterations'])
print(f"outer iterations      : {n_outer}")
print(f"lambda1 (signal model): {RECONSTRUCTION_PARAMS['lambda1']}")
print(f"lambda2 (SVD, l2)     : {RECONSTRUCTION_PARAMS['lambda2']}")
print(f"lambda3 (SVD, l1)     : {RECONSTRUCTION_PARAMS['lambda3']}")
print(f"parameter fitting     : {RECONSTRUCTION_PARAMS['fitting_opt_type']}")

<a id="2"></a>
## 2. Simulated data

**Data used here.** Numerical phantom generated with the XCAT framework: a short-axis
segmentation at the mid-ventricular level with the corresponding cardiac motion fields
$\boldsymbol{\phi}_{\text{xcat}}$ at 100 bpm. Tissue-specific quantitative parameters are
assigned to the segmentation and smoothed, giving the ground-truth maps
$\mathbf{p}_{\text{true}}$. Coil sensitivity maps estimated from an in-vivo scan and reoriented
to the phantom geometry define the acquisition operator $\tilde{\mathbf{A}}$, so that

$$
\mathbf{y}_{\text{simul}} \;=\; \tilde{\mathbf{A}}\,\mathbf{M}_{\boldsymbol{\phi}_{\text{xcat}}}\,\mathbf{Q}(\mathbf{p}_{\text{true}}) \;+\; \boldsymbol{\epsilon},
$$

with $\boldsymbol{\epsilon}$ complex Gaussian noise of standard deviation $10^{-3}$ of the
maximum k-space magnitude. Acquisition parameters match the in-vivo protocol: 2D golden-angle
radial sampling, continuous readout of 2.3 s after a single non-selective inversion pulse,
flip angle 5°, TE 2.03 ms, TR 5 ms, FOV 320 × 320 mm², in-plane resolution 1.3 × 1.3 mm²,
slice thickness 8 mm, and $T = 54$ images per dynamic.

`load_xcat_data` builds the synthetic experiment:

1. the XCAT ground-truth, piecewise-constant maps $(M_0, \alpha, T_1)$ are pushed through the transient
   steady-state signal model to get the motion-free dynamics $x^\text{true}$;
2. the *true* XCAT motion fields warp them into moving dynamics;
3. the moving dynamics are sampled with the real acquisition operator (trajectory and
   coil maps taken from an in-vivo scan) and complex Gaussian noise is added.

Returned quantities:

- `kdata_dynamic_simul` — simulated k-space, one dynamic per time frame,
- `rr_idx` — cardiac-phase (RR-interval) index of each dynamic, from the ECG,
- `p_true` — ground-truth, piecewise-constant $(M_0, \alpha[\deg], T_1[s])$, only used for the final figure,
- `sampling_time`, `repetition_time`, `tau`, `nominal_fa` — signal-model parameters.

In [ ]:
(kdata_vol,
 kdata_dynamic_simul,
 rr_idx,
 p_true,
 sampling_time, repetition_time, tau, nominal_fa) = load_xcat_data(RECONSTRUCTION_PARAMS, RUN_SETTINGS, device)

print(f'simulated k-space : {tuple(kdata_dynamic_simul.data.shape)}')
print(f'dynamics          : {len(rr_idx)}   (RR index range: {rr_idx.min().item()}-{rr_idx.max().item()})')

<a id="3"></a>
## 3. Operators and initial reconstruction

`construct_reconstruction_operators` returns the pieces of the forward model:

- `acquisition_op` $= \mathbf{A}$ — coil sensitivities + non-uniform FFT along the trajectory,
- `dcf_op` $= \mathbf{W}$ — density compensation, `dcf_sqrt_op` $= \mathbf{W}^{1/2}$,
- `AHWA_op` $= \mathbf{A}^H \mathbf{W} \mathbf{A}$ — the normal operator used inside CG.

Solving $\mathbf{A}^H \mathbf{W} \mathbf{A}\, \mathbf{x} = \mathbf{A}^H \mathbf{W} \mathbf{y}$ with a few CG iterations gives $\mathbf{x}_\text{m}$: the
undersampled **moving** dynamics. This is the starting point of the
alternating optimization.

In [ ]:
dcf_op, dcf_sqrt_op, acquisition_op, AHWA_op, _, img_size = construct_reconstruction_operators(
    kdata_dynamic=kdata_dynamic_simul,
    kdata=kdata_vol,
    device=device,
)

(rhs,) = (acquisition_op.H @ dcf_op)(kdata_dynamic_simul.data)

with torch.no_grad():
    x_moving, = mrpro.algorithms.optimizers.cg(
        AHWA_op,
        rhs,
        initial_value=rhs,
        max_iterations=RECONSTRUCTION_PARAMS['cg_iterations_initial'],
        tolerance=RECONSTRUCTION_PARAMS['cg_tolerance'],
    )

num_dynamics = x_moving.shape[0]
torch.cuda.empty_cache()

print(f'x_moving     : {tuple(x_moving.shape)}')
print(f'num_dynamics : {num_dynamics}')
print(f'image size   : {img_size}')

<a id="4"></a>
## 4. T1 estimator and No MoCo baseline

`T1ParameterEstimator` wraps the transient steady-state inversion-recovery model
$\mathbf{Q}(\mathbf{p})$ together with a `ConstraintsOp` that keeps $T_1$ and $\alpha$ in physical
ranges, and fits $\mathbf{p}$ by L-BFGS.

Fitted directly to the moving dynamics $\mathbf{x}_{\text{m}}$, over all $T$ frames, it returns the
baseline T1 map without any motion correction.

Since the acquisition is continuous, these frames span the full cardiac motion: voxels that
move across tissue boundaries are fitted from a mix of signals, and the T1 map degrades
exactly where motion is largest — the myocardial borders and the papillary muscles.

The same map is reused in the final comparison of [section 9](#9).

In [ ]:
t1_estimator = T1ParameterEstimator(
    sampling_time=sampling_time,
    repetition_time=repetition_time,
    nominal_flip_angle=nominal_fa,
    device=device,
    opt_type=RECONSTRUCTION_PARAMS['fitting_opt_type'],
)

# Baseline: signal model fitted directly to the moving (uncorrected) dynamics, all T frames.
p_no_moco, _ = t1_estimator.fit(x_moving, recon_params=RECONSTRUCTION_PARAMS)

T1_CLIM = (0, 2000)  # ms
t1_no_moco = roi_mask.apply(p_no_moco[-1].squeeze().abs()).detach().cpu().numpy() * 1e3


def prepare_image(img):
    return np.rot90(np.flipud(img), k=-1)  # flip vertically, then rotate 90 deg clockwise


fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(prepare_image(t1_no_moco), cmap=cmap_t1, clim=T1_CLIM)
ax.set_title('No MoCo')
ax.axis('off')
fig.colorbar(im, ax=ax, fraction=0.046, label='T1 [ms]')
plt.show()

<a id="5"></a>
## 5. Motion-compensated reference

To guide the registration we need a **motion-compensated target**. `find_valid_rr_threshold`
selects only the dynamics that fall in (almost) the same cardiac phase — those whose
`rr_idx` matches the first dynamic, progressively expanding the tolerance
(`[0] → [0,1] → [0,1,-1] → …`) until the fit converges without NaNs. Fitting $\mathbf{p}^{(0)}_{\text{r}}$ on that
subset and re-synthesizing $\mathbf{Q}(\mathbf{p}^{(0)}_\text{r})$ over all time points gives an initial motion-compensated
reference dynamic $\mathbf{x}^{(0)}_{\text{r}}$ that acts as the fixed image for registration.

In [ ]:
p_0_ref, x_0_ref, rr_expansion_offsets = find_valid_rr_threshold(
    rr_idx, x_moving, t1_estimator,
    initial_expansion=RECONSTRUCTION_PARAMS['initial_expansion'],
)
assert p_0_ref is not None, 'Could not find a valid RR threshold.'

# Cardiac-phase selection is progressively relaxed over the outer iterations.
base_rr_idx = rr_idx[0].item()
current_rr_expansion = rr_expansion_offsets.copy()

print(f'reference built with RR expansion {rr_expansion_offsets} (base index {base_rr_idx})')

<a id="6"></a>
## 6. Motion estimation

`MotionOperator` holds the forward warp $\mathbf{M}_{\boldsymbol{\phi}}$ and its inverse $\mathbf{M}_{\boldsymbol{\phi}^{-1}}$ as
`GridSamplingOp`s; it is initialized with zero displacement and is updated in place by
the registration network.

`MotionEstimationWithInverse` is a small U-Net that predicts, for every dynamic, the
displacement field to the reference phase **and** its inverse. It is trained from
scratch on this subject only (*zero-shot*) and without ground-truth motion
(*unsupervised*), minimizing the loss function

$$\arg\min_{\boldsymbol{\theta}} \ \mathcal{L}\big(\boldsymbol{\phi}_{\boldsymbol{\theta}},\boldsymbol{\phi}^{-1}_{\boldsymbol{\theta}},\mathbf{x}_{\text{m}},\mathbf{x}_{\text{r}}\big)$$

`mask_roi` restricts the loss to the cardiac ROI.

In [ ]:
motion_op = MotionOperator(times=num_dynamics, img_size=img_size, device=device)

motion_estimation = MotionEstimationWithInverse(
    kdata_dynamic=kdata_dynamic_simul.to(device),
    acquisition_op=acquisition_op.to(device),
    dcf_sqrt_op=dcf_sqrt_op.to(device),
    motion_op=motion_op,
    img_size=img_size,
    times=num_dynamics,
    lr=MOTION_NET_PARAMS['lr_phi'],
    weight_decay=MOTION_NET_PARAMS['weight_phi'],
    optimizer=MOTION_NET_PARAMS['optimizer'],
    mask_roi=roi_mask,
    ch_out_unet=MOTION_NET_PARAMS['ch_out_unet_phi'],
    n_enc_stages=MOTION_NET_PARAMS['n_enc_stages_phi'],
    n_filters=MOTION_NET_PARAMS['n_filters_phi'],
    weight_dc=MOTION_NET_PARAMS['weight_dc'],
    weight_mi=MOTION_NET_PARAMS['weight_mi'],
    lambd=MOTION_NET_PARAMS['lambd'],
    mu=MOTION_NET_PARAMS['mu'],
    device=device,
)

In [ ]:
# Initial registration. train_net(imgs_m, imgs_f): the moving dynamic and the fixed (reference) dynamic.
motion_estimation.train_net(
    x_moving.abs(),
    x_0_ref,
    episodes=MOTION_NET_PARAMS['phi_episodes_start'],
    steps=MOTION_NET_PARAMS['phi_steps_start'],
    full_loss_phi=MOTION_NET_PARAMS['full_loss_phi_start'],
)
torch.cuda.empty_cache()
print('\ninitial motion fields estimated')

<a id="7"></a>
## 7. Initialization (iteration 0)

Three things are initialized before the loop starts:

**$\mathbf{x}^{(0)}$** — apply the inverse motion to the moving dynamics, $\mathbf{x}^{(0)} = \mathbf{M}_{\boldsymbol{\phi}^{-1}} \mathbf{x}_\text{m}$,
so that all frames are brought back to the common reference phase.

**$\mathbf{p}^{(0)}$, $\mathbf{Q}(\mathbf{p}^{(0)})$** — fit the signal model on $\mathbf{x}^{(0)}$. Even after motion
correction, the frames acquired near peak systole are the least reliable, so
`select_sinusoidal_mask` fits a sinusoid to the mean motion amplitude over time and
drops the top 15 % (`dyns_selection_percentage = 0.85`) of the curve.

**SVD basis $\mathbf{U}$** — a temporal low-rank basis computed on the ROI of $\mathbf{x}^{(0)}$.

In [ ]:
# x^(0): motion-corrected dynamics
x_registered, = motion_op.M_op_inv(x_moving)
x_current = x_registered.detach()
print(f'x_current: {tuple(x_current.shape)}')

# Drop the 15% most-deformed (systolic) dynamics from the parameter fit.
optimal_phi = MotionOperator.normalized_grid_to_field(motion_op.M_op.grid)
dyn_mask = T1ParameterEstimator.select_sinusoidal_mask(
    optimal_phi.detach().cpu(),
    percentage=RECONSTRUCTION_PARAMS['dyns_selection_percentage'],
    plot=False,
)
if dyn_mask is not None:
    print(f'dynamics kept for fitting: {dyn_mask.sum()}/{len(dyn_mask)}')

# p^(0) and q(p^(0))
p, qp = t1_estimator.fit(x_current, mask=dyn_mask, recon_params=RECONSTRUCTION_PARAMS)

# Temporal SVD basis, computed once on the ROI.
x_reg_svd, U_op, _ = compute_svd(x_current, roi_mask, divide_by_n=RECONSTRUCTION_PARAMS['divide_by_n'])
x_reg_svd = x_reg_svd.detach().unsqueeze(1).unsqueeze(1)
print(f'p: {tuple(p.shape)} | qp: {tuple(qp.shape)} | x_reg_svd: {tuple(x_reg_svd.shape)}')

<a id="8"></a>
## 8. Alternating optimization loop

Each outer iteration cycles through the three blocks.

**(a) Motion fields $\boldsymbol{\phi}$, $\boldsymbol{\phi}^{-1}$.** The cardiac-phase window used to build the reference is
widened by one RR bin, a new $\mathbf{x}_\text{r}$ is fitted on it, and the registration network
is fine-tuned against it. Widening the window means more dynamics — hence more inversion
times and a better-conditioned T1 fit — which is only possible because the images are by
now partly motion-corrected. If the wider fit produces NaNs we fall back to the previous
reference.

**(b) Motion-corrected dynamic $\mathbf{x}$.** With $\boldsymbol{\phi}$ and $\mathbf{p}$ fixed, the objective is quadratic in $\mathbf{x}$ and its
normal equation

$$
\big(\mathbf{M}_{\boldsymbol{\phi}}^{H}\mathbf{A}^{H}\mathbf{W}\mathbf{A}\mathbf{M}_{\boldsymbol{\phi}} + \lambda_1 \mathrm{Id} + \lambda_2 \mathrm{Id}\big)\, \mathbf{x}
= \mathbf{M}_{\boldsymbol{\phi}}^{H}\mathbf{A}^{H}\mathbf{W}\mathbf{y} + \lambda_1\, \mathbf{Q}(\mathbf{p}) + \lambda_2\, \mathbf{U}\mathbf{U}^{H}\mathbf{x}
$$

is solved with a few CG iterations, warm-started from the previous $\mathbf{x}$. The SVD term on
the right-hand side is refreshed first by `update_svd` (soft-thresholding with weights
`lambda3` / `lambda2`).

**(c) Parameters $\mathbf{p}$.** Refit the signal model on the updated $\mathbf{x}$, again excluding the
systolic frames.

In [ ]:
for iteration in range(1, n_outer):
    print(f'\n{"-" * 40}\niteration {iteration}/{n_outer - 1}')

    # ---- (a) update the reference and the motion fields ------------------------------
    mask_ref, current_rr_expansion = update_rr_expansion_and_compute_rr_mask(
        rr_idx, base_rr_idx, current_rr_expansion
    )
    n_ref = mask_ref.sum().item()
    print(f'  x_ref : RR expansion {current_rr_expansion} -> {n_ref} dynamics '
          f'({n_ref / mask_ref.numel() * 100:.1f}%)')

    if mask_ref.any():
        p_ref_updated, x_ref_updated = t1_estimator.fit(
            x_current, mask=mask_ref, recon_params=RECONSTRUCTION_PARAMS
        )
        if torch.isnan(x_ref_updated).any():
            print('  x_ref contains NaN -> keeping the previous reference')
            p_ref_updated, x_ref_updated = p_0_ref, x_0_ref
    else:
        p_ref_updated, x_ref_updated = p_0_ref, x_0_ref

    motion_estimation.train_net(
        x_moving.abs(),
        x_ref_updated.detach(),
        episodes=MOTION_NET_PARAMS['phi_episodes_iter'],
        steps=MOTION_NET_PARAMS['phi_steps_iter'],
        full_loss_phi=MOTION_NET_PARAMS['full_loss_phi_iter'],
    )
    print()  # train_net prints with a carriage return
    optimal_phi = MotionOperator.normalized_grid_to_field(motion_op.M_op.grid)

    # ---- (b) update the images with CG -----------------------------------------------
    x_reg_svd, _ = update_svd(
        x_current, U_op,
        weight_l1=RECONSTRUCTION_PARAMS['lambda3'],
        weight_l2=RECONSTRUCTION_PARAMS['lambda2'],
        divide_by_n=RECONSTRUCTION_PARAMS['divide_by_n'],
    )

    with torch.no_grad():
        H = lambda y: (
            motion_op.adjoint(AHWA_op(motion_op.forward(y)[0])[0])[0]
            + RECONSTRUCTION_PARAMS['lambda1'] * y
            + RECONSTRUCTION_PARAMS['lambda2'] * y,
        )
        b = (
            motion_op.adjoint(acquisition_op.adjoint(dcf_op(kdata_dynamic_simul.data)[0])[0])[0]
            + RECONSTRUCTION_PARAMS['lambda1'] * qp
            + RECONSTRUCTION_PARAMS['lambda2'] * x_reg_svd
        )
        x_next, = mrpro.algorithms.optimizers.cg(
            operator=H,
            right_hand_side=b,
            initial_value=x_current,
            max_iterations=RECONSTRUCTION_PARAMS['cg_iterations_opt_pb'],
            tolerance=RECONSTRUCTION_PARAMS['cg_tolerance'],
        )

    x_current = x_next.detach()
    torch.cuda.empty_cache()

    # ---- (c) update the parameter maps -----------------------------------------------
    p_init = p.detach() if RECONSTRUCTION_PARAMS['p_init'] == 'previous' else p_ref_updated.detach()

    dyn_mask = T1ParameterEstimator.select_sinusoidal_mask(
        optimal_phi.detach().cpu(),
        percentage=RECONSTRUCTION_PARAMS['dyns_selection_percentage'],
        plot=False,
    )
    p, qp = t1_estimator.fit(
        x_current, mask=dyn_mask, initial_p=p_init, recon_params=RECONSTRUCTION_PARAMS
    )
    torch.cuda.empty_cache()

print('\nOptimization finished.')

<a id="9"></a>
## 9. Final T1 map

`p` stacks $(M_0, \alpha, T_1)$, so `p[-1]` is the T1 map in **seconds**; it is converted
to milliseconds for display and cropped to the cardiac ROI.

Three maps are shown side by side:

- **No MoCo** — the uncorrected baseline of [section 4](#4);
- **Proposed** — the motion-corrected T1 map returned by the iterative scheme;
- **Reference** — the ground truth from $\mathbf{p}_{\text{true}}$, i.e. the tissue-specific
  values assigned to the XCAT segmentation and smoothed with a Gaussian kernel to soften the
  sharp, unrealistic tissue boundaries.

In [ ]:
t1_estimated = roi_mask.apply(p[-1].squeeze().abs()).detach().cpu().numpy() * 1e3
t1_ground_truth = roi_mask.apply(p_true[-1].squeeze().abs()).detach().cpu().numpy() * 1e3

t1_ground_truth = gaussian_filter(
    t1_ground_truth,
    sigma=0.75,      # very light blur
)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))
for ax, img, title in zip(axes, [t1_no_moco, t1_estimated, t1_ground_truth], ['No MoCo', 'Proposed', 'Reference']):
    img_plot = prepare_image(img)
    im = ax.imshow(img_plot, cmap=cmap_t1, clim=T1_CLIM)
    ax.set_title(title)
    ax.axis('off')

fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.03, label='T1 [ms]')
fig.suptitle('XCAT Simulated Data')
plt.show()